# Ranking LLMs on Structured Data Processing

This notebook benchmarks and ranks large language models on **structured data processing** tasks built from the
AI4I 2020 predictive-maintenance data used elsewhere in this repository (machine type, temperatures, rotational
speed, torque, tool wear and failure flags).

Every task item is generated from the data with a **programmatically computed ground truth**, so scoring is
fully automatic: no LLM-as-judge, no hand labels.

| # | Module | What it does |
|---|--------|--------------|
| 1 | **Configuration** | Seeds, sample sizes, task weights, output paths |
| 2 | **Data** | Loads the AI4I 2020 dataset (local CSV → KaggleHub → deterministic synthetic fallback) and normalises columns |
| 3 | **Task suite** | Five task families: log → JSON extraction, table QA, CSV → JSON transformation, text-to-SQL, rule-based failure classification |
| 4 | **Model registry & adapters** | Anthropic (official SDK) and any OpenAI-compatible endpoint (OpenAI, Gemini, OpenRouter, Groq, Together, Ollama, vLLM…), plus calibration baselines |
| 5 | **Execution engine** | Parallel calls, on-disk response cache, latency / token / cost accounting |
| 6 | **Scoring** | Robust JSON extraction, JSON-Schema validation, field-level and set-based metrics, sandboxed SQL execution |
| 7 | **Ranking** | Weighted composite score, stratified bootstrap 95% CIs, rank probabilities, Bradley–Terry (Elo-scale) pairwise ratings, tiers |
| 8 | **Reporting** | Leaderboard, task heatmap, cost/quality Pareto chart, CSV / Markdown export |

**Running it.** With no API keys set, only the calibration baselines run, which checks that the pipeline and scorers
behave (oracle ≈ 1.0, null ≈ 0.0). Set one or more of the keys listed in Module 4 to benchmark real models.
Responses are cached in `llm_bench_cache/`, so re-running the notebook does not re-bill you.

In [ ]:
# %pip install -q pandas numpy matplotlib jsonschema anthropic openai kagglehub

## Module 1 — Configuration

In [ ]:
import hashlib
import json
import math
import os
import random
import re
import sqlite3
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable

import jsonschema
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 42
ITEMS_PER_TASK = 20          # items generated per task family (raise for tighter CIs)
MAX_WORKERS = 4              # concurrent requests per model
MAX_OUTPUT_TOKENS = 4096
REQUEST_TIMEOUT_S = 180
BOOTSTRAP_SAMPLES = 1000
INCLUDE_BASELINES = True     # show calibration baselines in the leaderboard

# Relative importance of each task family in the composite score (normalised automatically).
TASK_WEIGHTS = {
    "log_extraction": 1.0,
    "table_qa": 1.0,
    "csv_to_json": 1.0,
    "text_to_sql": 1.0,
    "failure_classification": 1.0,
}

CACHE_DIR = Path("llm_bench_cache")
RESULTS_DIR = Path("llm_bench_results")
CACHE_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.width", 160)

## Module 2 — Data

Loads the AI4I 2020 predictive-maintenance dataset. Lookup order:

1. a local CSV (`AI4I_CSV_PATH` env var, or `ai4i2020.csv` / `predictive_maintenance.csv` next to the notebook);
2. KaggleHub (same dataset as the failure-prediction notebook in this repo);
3. a deterministic synthetic generator with the same schema and the published failure rules, so the benchmark is
   reproducible offline.

Column names are normalised to snake_case so the same names appear in prompts, SQL schema and scorers.

In [ ]:
CANONICAL_COLUMNS = {
    "udi": ["udi"],
    "product_id": ["product id", "product_id"],
    "type": ["type"],
    "air_temp_k": ["air temperature"],
    "process_temp_k": ["process temperature"],
    "rpm": ["rotational speed"],
    "torque_nm": ["torque"],
    "tool_wear_min": ["tool wear"],
    "machine_failure": ["machine failure", "target"],
    "twf": ["twf"], "hdf": ["hdf"], "pwf": ["pwf"], "osf": ["osf"], "rnf": ["rnf"],
}


def normalise_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename = {}
    for col in df.columns:
        low = col.lower().strip()
        for canon, keys in CANONICAL_COLUMNS.items():
            if canon not in rename.values() and any(low == k or low.startswith(k) for k in keys):
                rename[col] = canon
                break
    df = df.rename(columns=rename)
    missing = {"product_id", "type", "air_temp_k", "process_temp_k", "rpm", "torque_nm", "tool_wear_min"} - set(df.columns)
    if missing:
        raise ValueError(f"Dataset is missing required columns: {sorted(missing)}")
    if "udi" not in df.columns:
        df.insert(0, "udi", range(1, len(df) + 1))
    for flag in ["machine_failure", "twf", "hdf", "pwf", "osf", "rnf"]:
        if flag not in df.columns:
            df[flag] = 0
    return df[list(CANONICAL_COLUMNS)]


def synthetic_ai4i(n: int = 10_000, seed: int = SEED) -> pd.DataFrame:
    """AI4I-2020-shaped data generated from the dataset's published distributions and failure rules."""
    rng = np.random.default_rng(seed)
    types = rng.choice(["L", "M", "H"], size=n, p=[0.6, 0.3, 0.1])
    air = np.round(rng.normal(300.0, 2.0, n), 1)
    process = np.round(air + 10.0 + rng.normal(0.0, 1.0, n), 1)
    rpm = np.clip(np.round(rng.normal(1540, 180, n)), 1168, 2886).astype(int)
    torque = np.round(np.clip(rng.normal(40.0, 10.0, n), 3.8, 76.6), 1)
    wear = rng.integers(0, 254, n)
    df = pd.DataFrame({
        "udi": np.arange(1, n + 1),
        "product_id": [f"{t}{rng.integers(10000, 99999)}" for t in types],
        "type": types, "air_temp_k": air, "process_temp_k": process,
        "rpm": rpm, "torque_nm": torque, "tool_wear_min": wear,
    })
    modes = df.apply(failure_modes, axis=1)
    for m in ["TWF", "HDF", "PWF", "OSF"]:
        df[m.lower()] = modes.apply(lambda s, m=m: int(m in s))
    df["rnf"] = (rng.random(n) < 0.001).astype(int)
    df["machine_failure"] = df[["twf", "hdf", "pwf", "osf", "rnf"]].max(axis=1)
    return df[list(CANONICAL_COLUMNS)]


# Failure-mode rules (AI4I 2020 documentation; TWF made deterministic so ground truth is well defined).
OSF_LIMIT = {"L": 11_000, "M": 12_000, "H": 13_000}


def power_w(row) -> float:
    return row["torque_nm"] * row["rpm"] * 2 * math.pi / 60


def failure_modes(row) -> list[str]:
    modes = []
    if row["tool_wear_min"] >= 200:
        modes.append("TWF")
    if (row["process_temp_k"] - row["air_temp_k"]) < 8.6 and row["rpm"] < 1380:
        modes.append("HDF")
    p = power_w(row)
    if p < 3500 or p > 9000:
        modes.append("PWF")
    if row["tool_wear_min"] * row["torque_nm"] > OSF_LIMIT[row["type"]]:
        modes.append("OSF")
    return modes


def load_ai4i() -> tuple[pd.DataFrame, str]:
    candidates = [os.environ.get("AI4I_CSV_PATH"), "ai4i2020.csv", "predictive_maintenance.csv"]
    for path in filter(None, candidates):
        if Path(path).is_file():
            return normalise_columns(pd.read_csv(path)), f"local CSV ({path})"
    try:
        import kagglehub
        folder = Path(kagglehub.dataset_download("abdelazizsami/predictive-maintenance-dataset"))
        csv = next(folder.glob("*.csv"))
        return normalise_columns(pd.read_csv(csv)), f"KaggleHub ({csv.name})"
    except Exception as exc:  # no network, no kagglehub, or no credentials
        print(f"KaggleHub unavailable ({type(exc).__name__}); using synthetic AI4I-shaped data.")
        return synthetic_ai4i(), "synthetic (seeded)"


df, DATA_SOURCE = load_ai4i()
df["type"] = df["type"].astype(str).str.strip().str.upper()
print(f"Data source: {DATA_SOURCE} — {len(df):,} rows")
df.head()

## Module 3 — Task suite

Each task family turns rows of the dataset into prompts with an exact expected answer. All models get the same
items, the same system prompt, and are asked to answer **with JSON only** (no provider-specific structured-output
mode is used, so format discipline is part of what is measured).

| Task | Skill tested | Expected output |
|------|--------------|-----------------|
| `log_extraction` | Pull typed fields out of free-text maintenance logs, incl. unit conversion (°C/°F → K) | JSON object matching a JSON Schema |
| `table_qa` | Filter / aggregate / argmax over a CSV table | `{"answer": ...}` |
| `csv_to_json` | Filter rows, derive a computed column, sort, reshape to JSON | JSON array of objects |
| `text_to_sql` | Translate a question to SQLite over the `machines` table | `{"sql": "..."}` — executed and compared on result sets |
| `failure_classification` | Apply multi-condition business rules to sensor rows | JSON array of failure-mode sets |

In [ ]:
SYSTEM_PROMPT = (
    "You are a precise data-processing engine. Follow the instructions exactly. "
    "Respond with valid JSON only: no prose, no explanations, no markdown code fences."
)


@dataclass
class TaskItem:
    item_id: str
    task: str
    prompt: str
    expected: Any
    meta: dict = field(default_factory=dict)


def table_csv(rows: pd.DataFrame, cols: list[str]) -> str:
    return rows[cols].to_csv(index=False).strip()


def balanced_sample(rng: random.Random, n: int, positive_frac: float = 0.5) -> pd.DataFrame:
    """Sample rows with a controlled share that trigger at least one failure-mode rule."""
    has_mode = df.apply(lambda r: bool(failure_modes(r)), axis=1)
    k_pos = min(int(round(n * positive_frac)), int(has_mode.sum()))
    pos = df[has_mode].sample(k_pos, random_state=rng.randrange(1 << 30))
    neg = df[~has_mode].sample(n - k_pos, random_state=rng.randrange(1 << 30))
    return pd.concat([pos, neg]).sample(frac=1, random_state=rng.randrange(1 << 30))


# ---------------------------------------------------------------- 3.1 log extraction
LOG_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": ["machine_id", "machine_type", "air_temp_k", "process_temp_k", "rpm",
                 "torque_nm", "tool_wear_min", "failure_reported"],
    "properties": {
        "machine_id": {"type": "string"},
        "machine_type": {"enum": ["L", "M", "H"]},
        "air_temp_k": {"type": "number"},
        "process_temp_k": {"type": "number"},
        "rpm": {"type": "integer"},
        "torque_nm": {"type": "number"},
        "tool_wear_min": {"type": "integer"},
        "failure_reported": {"type": "boolean"},
    },
}
TYPE_WORDS = {"L": ["low-grade", "L-variant", "economy (L)"],
              "M": ["medium-grade", "M-variant", "standard (M)"],
              "H": ["high-grade", "H-variant", "premium (H)"]}


def fmt_temp(kelvin: float, rng: random.Random) -> str:
    unit = rng.choice(["K", "C", "F"])
    if unit == "K":
        return f"{kelvin:.1f} K"
    if unit == "C":
        return f"{kelvin - 273.15:.2f} °C"
    return f"{(kelvin - 273.15) * 9 / 5 + 32:.2f} °F"


def gen_log_extraction(rng: random.Random, n: int) -> list[TaskItem]:
    items = []
    for i, (_, r) in enumerate(balanced_sample(rng, n, 0.4).iterrows()):
        failed = bool(failure_modes(r)) or bool(r["machine_failure"])
        parts = [
            f"unit {r['product_id']} ({rng.choice(TYPE_WORDS[r['type']])})",
            f"ambient air read {fmt_temp(r['air_temp_k'], rng)}",
            f"process side at {fmt_temp(r['process_temp_k'], rng)}",
            rng.choice([f"spindle turning {r['rpm']} rpm", f"speed {r['rpm']} rev/min"]),
            rng.choice([f"torque {r['torque_nm']} Nm", f"load torque of {r['torque_nm']} N·m"]),
            rng.choice([f"tool has {r['tool_wear_min']} min of wear", f"cutter wear {r['tool_wear_min']} minutes"]),
            rng.choice(["operator flagged a breakdown", "machine stopped with a fault"]) if failed
            else rng.choice(["no fault observed", "ran normally"]),
        ]
        head, rest = parts[0], parts[1:]
        rng.shuffle(rest)
        log = f"[shift log] Checked {head}: " + "; ".join(rest) + "."
        expected = {
            "machine_id": r["product_id"], "machine_type": r["type"],
            "air_temp_k": round(float(r["air_temp_k"]), 1), "process_temp_k": round(float(r["process_temp_k"]), 1),
            "rpm": int(r["rpm"]), "torque_nm": float(r["torque_nm"]), "tool_wear_min": int(r["tool_wear_min"]),
            "failure_reported": failed,
        }
        prompt = (
            "Extract the machine reading from this maintenance log into a JSON object that conforms to this JSON Schema:\n"
            f"{json.dumps(LOG_SCHEMA)}\n\n"
            "Rules: temperatures must be in Kelvin rounded to 1 decimal (K = °C + 273.15; °C = (°F − 32) × 5/9); "
            "machine_type is the single letter L, M or H; failure_reported is true if the log reports a fault.\n\n"
            f"Log:\n{log}"
        )
        items.append(TaskItem(f"log_extraction-{i:03d}", "log_extraction", prompt, expected))
    return items


# ---------------------------------------------------------------- 3.2 table QA
QA_COLS = ["product_id", "type", "air_temp_k", "process_temp_k", "rpm", "torque_nm", "tool_wear_min", "machine_failure"]


def gen_table_qa(rng: random.Random, n: int) -> list[TaskItem]:
    items = []
    while len(items) < n:
        t = balanced_sample(rng, 25, 0.3)
        mtype = rng.choice(["L", "M", "H"])
        wear_t = rng.choice([50, 100, 150, 200])
        kind = rng.choice(["mean_torque", "count_wear", "max_rpm_fail", "argmax_torque", "temp_delta"])
        if kind == "mean_torque":
            sub = t[t["type"] == mtype]
            if sub.empty:
                continue
            q, ans = f"What is the mean torque_nm of machines with type {mtype}? Round to 2 decimals.", round(sub["torque_nm"].mean(), 2)
        elif kind == "count_wear":
            q, ans = f"How many machines have tool_wear_min strictly greater than {wear_t}?", int((t["tool_wear_min"] > wear_t).sum())
        elif kind == "max_rpm_fail":
            sub = t[t["machine_failure"] == 1]
            if sub.empty:
                continue
            q, ans = "What is the highest rpm among rows with machine_failure = 1?", int(sub["rpm"].max())
        elif kind == "argmax_torque":
            if (t["torque_nm"] == t["torque_nm"].max()).sum() > 1:
                continue
            q, ans = "Which product_id has the highest torque_nm?", t.loc[t["torque_nm"].idxmax(), "product_id"]
        else:
            delta = (t["process_temp_k"] - t["air_temp_k"])
            q, ans = ("What is the average of (process_temp_k − air_temp_k) across all rows? Round to 2 decimals.",
                      round(delta.mean(), 2))
        prompt = (f"Answer the question using this CSV table.\n\n{table_csv(t, QA_COLS)}\n\n"
                  f"Question: {q}\nReturn JSON of the form {{\"answer\": <value>}}.")
        items.append(TaskItem(f"table_qa-{len(items):03d}", "table_qa", prompt, ans, {"kind": kind}))
    return items


# ---------------------------------------------------------------- 3.3 CSV -> JSON transformation
def gen_csv_to_json(rng: random.Random, n: int) -> list[TaskItem]:
    items = []
    while len(items) < n:
        t = balanced_sample(rng, 15, 0.3)
        mtype = rng.choice(["L", "M", "H"])
        wear_t = rng.choice([60, 100, 140])
        cond_text = f"type is {mtype} or tool_wear_min > {wear_t}"
        sub = t[(t["type"] == mtype) | (t["tool_wear_min"] > wear_t)].copy()
        sub["power_w"] = sub.apply(lambda r: round(power_w(r)), axis=1)
        if sub.empty or sub["power_w"].duplicated().any():
            continue
        sub = sub.sort_values("power_w", ascending=False)
        expected = [{"product_id": r.product_id, "type": r.type, "power_w": int(r.power_w)} for r in sub.itertuples()]
        prompt = (
            "Transform this CSV.\n\n"
            f"{table_csv(t, ['product_id', 'type', 'rpm', 'torque_nm', 'tool_wear_min'])}\n\n"
            f"Keep only rows where {cond_text}. For each kept row compute mechanical power "
            "power_w = torque_nm × rpm × 2π / 60, rounded to the nearest integer. "
            "Return a JSON array of objects with exactly the keys product_id, type, power_w, sorted by power_w descending."
        )
        items.append(TaskItem(f"csv_to_json-{len(items):03d}", "csv_to_json", prompt, expected))
    return items


# ---------------------------------------------------------------- 3.4 text-to-SQL
SQL_COLUMNS = list(CANONICAL_COLUMNS)
SQL_SCHEMA = ("CREATE TABLE machines (udi INTEGER, product_id TEXT, type TEXT, air_temp_k REAL, process_temp_k REAL, "
              "rpm INTEGER, torque_nm REAL, tool_wear_min INTEGER, machine_failure INTEGER, twf INTEGER, "
              "hdf INTEGER, pwf INTEGER, osf INTEGER, rnf INTEGER);")
SQL_DB = sqlite3.connect(":memory:", check_same_thread=False)
df[SQL_COLUMNS].to_sql("machines", SQL_DB, index=False)


def run_sql(sql: str) -> list[tuple]:
    return SQL_DB.execute(sql).fetchall()


def gen_text_to_sql(rng: random.Random, n: int) -> list[TaskItem]:
    templates = [
        lambda p: (f"How many type {p['t']} machines had a machine failure?",
                   f"SELECT COUNT(*) FROM machines WHERE type = '{p['t']}' AND machine_failure = 1"),
        lambda p: ("For each machine type, what is the average torque_nm? Return type and average.",
                   "SELECT type, AVG(torque_nm) FROM machines GROUP BY type"),
        lambda p: (f"What is the failure rate (fraction of rows with machine_failure = 1) per type for rows with rpm below {p['rpm']}? Return type and rate.",
                   f"SELECT type, AVG(machine_failure) FROM machines WHERE rpm < {p['rpm']} GROUP BY type"),
        lambda p: (f"List the product_id and tool_wear_min of the {p['k']} type {p['t']} machines with the highest tool wear.",
                   f"SELECT product_id, tool_wear_min FROM machines WHERE type = '{p['t']}' ORDER BY tool_wear_min DESC LIMIT {p['k']}"),
        lambda p: (f"How many machines have tool_wear_min above {p['w']} and torque_nm above {p['q']}?",
                   f"SELECT COUNT(*) FROM machines WHERE tool_wear_min > {p['w']} AND torque_nm > {p['q']}"),
        lambda p: ("Which machine type has the most heat dissipation failures (hdf = 1)? Return the type and the count.",
                   "SELECT type, COUNT(*) AS n FROM machines WHERE hdf = 1 GROUP BY type ORDER BY n DESC LIMIT 1"),
        lambda p: (f"What is the average temperature difference process_temp_k minus air_temp_k for type {p['t']} machines that failed?",
                   f"SELECT AVG(process_temp_k - air_temp_k) FROM machines WHERE type = '{p['t']}' AND machine_failure = 1"),
    ]
    items = []
    while len(items) < n:
        p = {"t": rng.choice("LMH"), "rpm": rng.choice([1300, 1400, 1500]), "k": rng.choice([3, 5]),
             "w": rng.choice([150, 180, 200]), "q": rng.choice([40, 50, 60])}
        question, gold = rng.choice(templates)(p)
        if "LIMIT" in gold and "ORDER BY tool_wear_min" in gold:
            # Skip if a tie at the LIMIT boundary would make several answers correct.
            k = p["k"]
            wear = df[df["type"] == p["t"]]["tool_wear_min"].sort_values(ascending=False).tolist()
            if len(wear) > k and wear[k - 1] == wear[k]:
                continue
        prompt = (f"SQLite schema:\n{SQL_SCHEMA}\n\nWrite one SQLite SELECT query that answers: {question}\n"
                  "Return only the requested columns, in the order they are mentioned. "
                  "Return JSON of the form {\"sql\": \"<query>\"}.")
        items.append(TaskItem(f"text_to_sql-{len(items):03d}", "text_to_sql", prompt, run_sql(gold), {"gold_sql": gold}))
    return items


# ---------------------------------------------------------------- 3.5 rule-based failure classification
RULES_TEXT = """Failure-mode rules (a row can have several modes, or none):
- TWF (tool wear failure): tool_wear_min >= 200
- HDF (heat dissipation failure): (process_temp_k - air_temp_k) < 8.6 AND rpm < 1380
- PWF (power failure): power_w = torque_nm * rpm * 2*pi / 60; PWF if power_w < 3500 OR power_w > 9000
- OSF (overstrain failure): tool_wear_min * torque_nm > limit, where limit is 11000 for type L, 12000 for type M, 13000 for type H"""


def gen_failure_classification(rng: random.Random, n: int) -> list[TaskItem]:
    items = []
    for i in range(n):
        rows = balanced_sample(rng, 6, 0.5)
        records = rows[["product_id", "type", "air_temp_k", "process_temp_k", "rpm", "torque_nm", "tool_wear_min"]].to_dict("records")
        expected = [{"product_id": r["product_id"], "failure_modes": sorted(failure_modes(r))} for r in records]
        prompt = (f"{RULES_TEXT}\n\nSensor rows (JSON):\n{json.dumps(records, default=float)}\n\n"
                  "Apply the rules to every row. Return a JSON array with one object per row, in the same order, "
                  "with keys product_id and failure_modes (an array of mode codes, empty if none).")
        items.append(TaskItem(f"failure_classification-{i:03d}", "failure_classification", prompt, expected))
    return items


TASK_GENERATORS: dict[str, Callable[[random.Random, int], list[TaskItem]]] = {
    "log_extraction": gen_log_extraction,
    "table_qa": gen_table_qa,
    "csv_to_json": gen_csv_to_json,
    "text_to_sql": gen_text_to_sql,
    "failure_classification": gen_failure_classification,
}

ITEMS: list[TaskItem] = []
for name, gen in TASK_GENERATORS.items():
    ITEMS += gen(random.Random(f"{SEED}-{name}"), ITEMS_PER_TASK)

print(f"Generated {len(ITEMS)} items across {len(TASK_GENERATORS)} task families")
print("\nExample prompt (log_extraction):\n" + ITEMS[0].prompt)

## Module 4 — Model registry & adapters

* **Anthropic** models are called through the official `anthropic` SDK.
* **Every other provider** is reached through its OpenAI-compatible Chat Completions endpoint via the `openai` SDK,
  so one adapter covers OpenAI, Google Gemini, OpenRouter (hundreds of models behind one key), Groq, Together,
  Mistral, DeepSeek, xAI, local Ollama / vLLM / LM Studio and others.

A model is benchmarked only when its provider's API key is set. Non-Anthropic model IDs change often, so they are
supplied through environment variables rather than hard-coded, e.g.

```bash
export ANTHROPIC_API_KEY=...                       # Claude models below
export OPENAI_API_KEY=...;     export OPENAI_MODELS="model-a,model-b"
export GEMINI_API_KEY=...;     export GEMINI_MODELS="model-c"
export OPENROUTER_API_KEY=...; export OPENROUTER_MODELS="vendor/model-d,vendor/model-e"
export OLLAMA_MODELS="llama-local"                  # local, no key
```

Optional pricing (USD per 1M input/output tokens) for those models: `MODEL_PRICES='{"vendor/model-d": [0.5, 1.5]}'`.

Notes on fairness:
* No sampling parameters (temperature etc.) are sent, so each model runs at its provider default.
* Server-side refusal **fallbacks are deliberately not enabled** for Claude models: a fallback would let a
  different model answer and contaminate that model's score. Refusals are recorded as failed items instead.
* The **calibration baselines** are not LLMs: `oracle` returns the ground truth (should score 1.0),
  `noisy-oracle` corrupts ~30% of answers, `null` returns `{}` (should score ≈ 0). They verify the scorers and show
  the ranking separates good from bad answers.

In [ ]:
@dataclass
class ModelSpec:
    key: str                       # unique label used in the leaderboard
    provider: str                  # "anthropic" | "openai_compatible" | "baseline"
    model_id: str
    input_usd_per_mtok: float = float("nan")
    output_usd_per_mtok: float = float("nan")
    base_url: str | None = None
    api_key_env: str | None = None
    extra: dict = field(default_factory=dict)


@dataclass
class Completion:
    text: str
    input_tokens: int = 0
    output_tokens: int = 0
    latency_s: float = 0.0
    error: str | None = None


# Claude models: (model id, input $/MTok, output $/MTok). Override with ANTHROPIC_MODELS="id1,id2".
ANTHROPIC_DEFAULTS = [
    ("claude-fable-5-1", 10.00, 50.00),
    ("claude-opus-5-5", 4.00, 20.00),
    ("claude-sonnet-5-5", 2.00, 10.00),
    ("claude-haiku-5-5", 0.10, 0.50),
]

OPENAI_COMPATIBLE_PROVIDERS = {
    # env prefix: (base_url, api-key env var or None for keyless local servers)
    "OPENAI": ("https://api.openai.com/v1", "OPENAI_API_KEY"),
    "GEMINI": ("https://generativelanguage.googleapis.com/v1beta/openai/", "GEMINI_API_KEY"),
    "OPENROUTER": ("https://openrouter.ai/api/v1", "OPENROUTER_API_KEY"),
    "GROQ": ("https://api.groq.com/openai/v1", "GROQ_API_KEY"),
    "TOGETHER": ("https://api.together.xyz/v1", "TOGETHER_API_KEY"),
    "MISTRAL": ("https://api.mistral.ai/v1", "MISTRAL_API_KEY"),
    "DEEPSEEK": ("https://api.deepseek.com/v1", "DEEPSEEK_API_KEY"),
    "XAI": ("https://api.x.ai/v1", "XAI_API_KEY"),
    "OLLAMA": (os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434/v1"), None),
}


def env_list(name: str) -> list[str]:
    return [m.strip() for m in os.environ.get(name, "").split(",") if m.strip()]


def anthropic_credentials_available() -> bool:
    return bool(os.environ.get("ANTHROPIC_API_KEY") or os.environ.get("ANTHROPIC_AUTH_TOKEN")
                or os.environ.get("ANTHROPIC_PROFILE") or (Path.home() / ".config" / "anthropic").is_dir())


def build_registry() -> list[ModelSpec]:
    specs: list[ModelSpec] = []
    prices = json.loads(os.environ.get("MODEL_PRICES", "{}"))
    if anthropic_credentials_available():
        defaults = {m: (i, o) for m, i, o in ANTHROPIC_DEFAULTS}
        for model_id in env_list("ANTHROPIC_MODELS") or list(defaults):
            i, o = prices.get(model_id, defaults.get(model_id, (float("nan"), float("nan"))))
            specs.append(ModelSpec(model_id, "anthropic", model_id, i, o))
    for prefix, (base_url, key_env) in OPENAI_COMPATIBLE_PROVIDERS.items():
        if key_env and not os.environ.get(key_env):
            continue
        for model_id in env_list(f"{prefix}_MODELS"):
            i, o = prices.get(model_id, (float("nan"), float("nan")))
            specs.append(ModelSpec(f"{prefix.lower()}:{model_id}", "openai_compatible", model_id, i, o,
                                   base_url=base_url, api_key_env=key_env))
    if INCLUDE_BASELINES:
        specs += [ModelSpec(f"baseline:{b}", "baseline", b, 0.0, 0.0) for b in ["oracle", "noisy-oracle", "null"]]
    return specs


# ------------------------------------------------------------------ adapters
class AnthropicAdapter:
    def __init__(self, spec: ModelSpec):
        import anthropic
        self.anthropic = anthropic
        self.spec = spec
        self.client = anthropic.Anthropic(timeout=REQUEST_TIMEOUT_S, max_retries=3)

    def complete(self, item: TaskItem) -> Completion:
        a = self.anthropic
        t0 = time.perf_counter()
        try:
            resp = self.client.messages.create(
                model=self.spec.model_id,
                max_tokens=MAX_OUTPUT_TOKENS,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": item.prompt}],
                **self.spec.extra,  # e.g. {"output_config": {"effort": "low"}}
            )
        except a.RateLimitError as e:
            return Completion("", latency_s=time.perf_counter() - t0, error=f"rate_limited: {e.message}")
        except a.APIStatusError as e:
            return Completion("", latency_s=time.perf_counter() - t0, error=f"http_{e.status_code}: {e.message}")
        except a.APIConnectionError as e:
            return Completion("", latency_s=time.perf_counter() - t0, error=f"connection: {e}")
        latency = time.perf_counter() - t0
        text = "".join(b.text for b in resp.content if b.type == "text")
        error = "refusal" if resp.stop_reason == "refusal" else ("max_tokens" if resp.stop_reason == "max_tokens" else None)
        return Completion(text, resp.usage.input_tokens, resp.usage.output_tokens, latency, error)


class OpenAICompatibleAdapter:
    def __init__(self, spec: ModelSpec):
        import openai
        self.openai = openai
        self.spec = spec
        api_key = os.environ.get(spec.api_key_env) if spec.api_key_env else "not-needed"
        self.client = openai.OpenAI(base_url=spec.base_url, api_key=api_key, timeout=REQUEST_TIMEOUT_S, max_retries=3)

    def complete(self, item: TaskItem) -> Completion:
        t0 = time.perf_counter()
        try:
            resp = self.client.chat.completions.create(
                model=self.spec.model_id,
                max_completion_tokens=MAX_OUTPUT_TOKENS,
                messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": item.prompt}],
                **self.spec.extra,
            )
        except self.openai.APIStatusError as e:
            return Completion("", latency_s=time.perf_counter() - t0, error=f"http_{e.status_code}: {e.message}")
        except self.openai.APIConnectionError as e:
            return Completion("", latency_s=time.perf_counter() - t0, error=f"connection: {e}")
        latency = time.perf_counter() - t0
        choice = resp.choices[0] if resp.choices else None
        text = (choice.message.content or "") if choice else ""
        usage = resp.usage
        error = "max_tokens" if choice and choice.finish_reason == "length" else None
        return Completion(text, getattr(usage, "prompt_tokens", 0) or 0, getattr(usage, "completion_tokens", 0) or 0, latency, error)


class BaselineAdapter:
    """Non-LLM calibration baselines used to sanity-check scoring and ranking."""

    def __init__(self, spec: ModelSpec):
        self.spec = spec

    def complete(self, item: TaskItem) -> Completion:
        if self.spec.model_id == "null":
            return Completion("{}")
        answer = render_expected(item)
        if self.spec.model_id == "noisy-oracle":
            rng = random.Random(f"{SEED}-{item.item_id}")
            if rng.random() < 0.3:
                answer = corrupt(item, rng)
        return Completion(answer)


def render_expected(item: TaskItem) -> str:
    if item.task == "table_qa":
        return json.dumps({"answer": item.expected})
    if item.task == "text_to_sql":
        return json.dumps({"sql": item.meta["gold_sql"]})
    return json.dumps(item.expected)


def corrupt(item: TaskItem, rng: random.Random) -> str:
    exp = json.loads(render_expected(item))
    if item.task == "log_extraction":
        exp["rpm"] += rng.choice([-50, 50]); exp["failure_reported"] = not exp["failure_reported"]
    elif item.task == "table_qa":
        exp["answer"] = exp["answer"] * 1.1 if isinstance(exp["answer"], (int, float)) else "UNKNOWN"
    elif item.task == "csv_to_json":
        exp = exp[:-1] if len(exp) > 1 else exp + [{"product_id": "X0", "type": "L", "power_w": 1}]
    elif item.task == "text_to_sql":
        exp["sql"] = exp["sql"].replace("machine_failure = 1", "machine_failure = 0").replace("DESC", "ASC").replace("<", ">")
    elif item.task == "failure_classification":
        exp[0]["failure_modes"] = [] if exp[0]["failure_modes"] else ["TWF"]
    return json.dumps(exp)


ADAPTERS = {"anthropic": AnthropicAdapter, "openai_compatible": OpenAICompatibleAdapter, "baseline": BaselineAdapter}

REGISTRY = build_registry()
pd.DataFrame([{"model": s.key, "provider": s.provider, "in $/MTok": s.input_usd_per_mtok,
               "out $/MTok": s.output_usd_per_mtok} for s in REGISTRY])

## Module 5 — Execution engine

Runs every item against every registered model with a thread pool. Each response is cached on disk under a hash of
(model, system prompt, prompt), so interrupted or repeated runs only pay for missing items. Failed calls are not
cached and are retried on the next run.

In [ ]:
def cache_path(spec: ModelSpec) -> Path:
    return CACHE_DIR / (re.sub(r"[^A-Za-z0-9_.-]+", "_", spec.key) + ".jsonl")


def cache_key(spec: ModelSpec, item: TaskItem) -> str:
    payload = json.dumps([spec.provider, spec.model_id, spec.extra, SYSTEM_PROMPT, item.prompt], sort_keys=True)
    return hashlib.sha256(payload.encode()).hexdigest()


def load_cache(spec: ModelSpec) -> dict[str, dict]:
    path = cache_path(spec)
    if not path.exists():
        return {}
    with path.open() as f:
        return {rec["key"]: rec for rec in map(json.loads, f)}


def run_model(spec: ModelSpec, items: list[TaskItem]) -> list[dict]:
    adapter = ADAPTERS[spec.provider](spec)
    cache = {} if spec.provider == "baseline" else load_cache(spec)
    records, todo = [], []
    for item in items:
        k = cache_key(spec, item)
        (records.append(cache[k]) if k in cache else todo.append((k, item)))

    def call(k: str, item: TaskItem) -> dict:
        c = adapter.complete(item)
        return {"key": k, "model": spec.key, "item_id": item.item_id, "text": c.text, "input_tokens": c.input_tokens,
                "output_tokens": c.output_tokens, "latency_s": c.latency_s, "error": c.error}

    if todo:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool, cache_path(spec).open("a") as f:
            futures = [pool.submit(call, k, it) for k, it in todo]
            for fut in as_completed(futures):
                rec = fut.result()
                records.append(rec)
                if spec.provider != "baseline" and not (rec["error"] or "").startswith(("http_", "connection", "rate_limited")):
                    f.write(json.dumps(rec) + "\n")
    print(f"{spec.key:40s} {len(items) - len(todo):4d} cached, {len(todo):4d} new")
    return records


RAW: list[dict] = []
for spec in REGISTRY:
    RAW += run_model(spec, ITEMS)

## Module 6 — Scoring

Every response gets a score in **[0, 1]** plus diagnostics:

* `parsed` — a JSON value could be recovered from the response (fences / stray prose tolerated);
* `strict_json` — the whole response was valid JSON exactly as instructed;
* `schema_ok` — (log extraction) the object validates against the JSON Schema.

| Task | Score |
|------|-------|
| `log_extraction` | fraction of the 8 fields correct (temps ±0.15 K, torque ±0.05, ints/strings/bools exact); 0 if not an object |
| `table_qa` | 1 if the answer matches (numbers within 1% or ±0.01, strings case-insensitive) |
| `csv_to_json` | 0.6 × F1 on kept product_ids + 0.25 × share of matched rows with correct power_w (±1) & type + 0.15 × correct order |
| `text_to_sql` | 1 if the query runs read-only and its result set equals the gold result set (order-insensitive, floats rounded to 2 dp) |
| `failure_classification` | mean per-row Jaccard similarity between predicted and true failure-mode sets |

In [ ]:
FENCE_RE = re.compile(r"```(?:json)?\s*(.*?)```", re.S)


def extract_json(text: str) -> tuple[Any, bool, bool]:
    """Return (value, parsed, strict_json)."""
    text = (text or "").strip()
    try:
        return json.loads(text), True, True
    except (json.JSONDecodeError, TypeError):
        pass
    for block in FENCE_RE.findall(text):
        try:
            return json.loads(block.strip()), True, False
        except json.JSONDecodeError:
            continue
    decoder = json.JSONDecoder()
    for i, ch in enumerate(text):
        if ch in "{[":
            try:
                return decoder.raw_decode(text[i:])[0], True, False
            except json.JSONDecodeError:
                continue
    return None, False, False


def num_close(a, b, rel=0.01, abs_tol=0.01) -> bool:
    try:
        return math.isclose(float(a), float(b), rel_tol=rel, abs_tol=abs_tol)
    except (TypeError, ValueError):
        return False


def score_log_extraction(pred, item):
    if not isinstance(pred, dict):
        return 0.0, {"schema_ok": False}
    schema_ok = jsonschema.Draft202012Validator(LOG_SCHEMA).is_valid(pred)
    exp = item.expected
    checks = {
        "machine_id": str(pred.get("machine_id", "")).strip() == exp["machine_id"],
        "machine_type": pred.get("machine_type") == exp["machine_type"],
        "air_temp_k": num_close(pred.get("air_temp_k"), exp["air_temp_k"], 0, 0.15),
        "process_temp_k": num_close(pred.get("process_temp_k"), exp["process_temp_k"], 0, 0.15),
        "rpm": num_close(pred.get("rpm"), exp["rpm"], 0, 0.5),
        "torque_nm": num_close(pred.get("torque_nm"), exp["torque_nm"], 0, 0.05),
        "tool_wear_min": num_close(pred.get("tool_wear_min"), exp["tool_wear_min"], 0, 0.5),
        "failure_reported": pred.get("failure_reported") is exp["failure_reported"],
    }
    return sum(checks.values()) / len(checks), {"schema_ok": schema_ok}


def score_table_qa(pred, item):
    ans = pred.get("answer") if isinstance(pred, dict) else pred
    exp = item.expected
    if isinstance(exp, str):
        return float(str(ans).strip().lower() == exp.lower()), {}
    return float(num_close(ans, exp)), {}


def score_csv_to_json(pred, item):
    if not isinstance(pred, list):
        return 0.0, {}
    exp = {r["product_id"]: r for r in item.expected}
    got = {str(r.get("product_id")): r for r in pred if isinstance(r, dict)}
    tp = len(exp.keys() & got.keys())
    if tp == 0:
        return 0.0, {}
    precision, recall = tp / len(got), tp / len(exp)
    f1 = 2 * precision * recall / (precision + recall)
    matched = exp.keys() & got.keys()
    values = sum(num_close(got[k].get("power_w"), exp[k]["power_w"], 0, 1.0) and got[k].get("type") == exp[k]["type"]
                 for k in matched) / len(matched)
    order_ok = [str(r.get("product_id")) for r in pred if isinstance(r, dict)] == [r["product_id"] for r in item.expected]
    return 0.6 * f1 + 0.25 * values + 0.15 * order_ok, {}


def _read_only(action, *_):
    allowed = {sqlite3.SQLITE_SELECT, sqlite3.SQLITE_READ, sqlite3.SQLITE_FUNCTION}
    return sqlite3.SQLITE_OK if action in allowed else sqlite3.SQLITE_DENY


def normalise_rows(rows) -> list[tuple]:
    norm = lambda v: round(float(v), 2) if isinstance(v, (int, float)) and not isinstance(v, bool) else str(v)
    return sorted((tuple(norm(v) for v in r) for r in rows), key=repr)


def score_text_to_sql(pred, item):
    sql = pred.get("sql") if isinstance(pred, dict) else None
    if not isinstance(sql, str) or not sql.strip():
        return 0.0, {"sql_error": "no sql"}
    conn = sqlite3.connect(":memory:")
    SQL_DB.backup(conn)                       # private copy, so a bad query can't affect other scores
    conn.set_authorizer(_read_only)
    try:
        rows = conn.execute(sql.strip().rstrip(";")).fetchmany(10_000)
    except sqlite3.Error as e:
        return 0.0, {"sql_error": str(e)[:120]}
    finally:
        conn.close()
    return float(normalise_rows(rows) == normalise_rows(item.expected)), {}


def score_failure_classification(pred, item):
    if not isinstance(pred, list):
        return 0.0, {}
    by_id = {str(r.get("product_id")): r for r in pred if isinstance(r, dict)}
    total = 0.0
    for exp in item.expected:
        got = by_id.get(exp["product_id"], {}).get("failure_modes", None)
        got = {str(m).upper() for m in got} if isinstance(got, list) else None
        truth = set(exp["failure_modes"])
        if got is None:
            continue
        total += 1.0 if not truth and not got else len(truth & got) / len(truth | got)
    return total / len(item.expected), {}


SCORERS = {
    "log_extraction": score_log_extraction,
    "table_qa": score_table_qa,
    "csv_to_json": score_csv_to_json,
    "text_to_sql": score_text_to_sql,
    "failure_classification": score_failure_classification,
}
ITEM_BY_ID = {it.item_id: it for it in ITEMS}
SPEC_BY_KEY = {s.key: s for s in REGISTRY}


def score_record(rec: dict) -> dict:
    item = ITEM_BY_ID[rec["item_id"]]
    value, parsed, strict = extract_json(rec["text"])
    score, extra = SCORERS[item.task](value, item) if parsed else (0.0, {})
    spec = SPEC_BY_KEY[rec["model"]]
    cost = (rec["input_tokens"] * spec.input_usd_per_mtok + rec["output_tokens"] * spec.output_usd_per_mtok) / 1e6
    return {"model": rec["model"], "provider": spec.provider, "task": item.task, "item_id": item.item_id,
            "score": score, "parsed": parsed, "strict_json": strict, "schema_ok": extra.get("schema_ok"),
            "error": rec["error"], "latency_s": rec["latency_s"], "input_tokens": rec["input_tokens"],
            "output_tokens": rec["output_tokens"], "cost_usd": cost}


results = pd.DataFrame([score_record(r) for r in RAW])
results.groupby(["model", "task"])["score"].mean().unstack().round(3)

## Module 7 — Ranking

* **Composite score** — weighted mean of per-task mean scores (weights from Module 1).
* **95% CI & rank probabilities** — stratified bootstrap over items: each replicate resamples items *within each
  task* (shared across models, so comparisons are paired), recomputes every model's composite and ranks them.
* **Bradley–Terry rating** — every item is a "match" between each pair of models (higher item score wins, equal
  scores tie). Strengths are fit with the standard MM algorithm and shown on an Elo-like scale (1000 = average).
* **Tier** — models whose CI overlaps the CI of the best model in the tier are grouped together; differences within a
  tier are not statistically meaningful at this sample size.

In [ ]:
weights = pd.Series(TASK_WEIGHTS, dtype=float)
weights = weights[weights.index.isin(results["task"].unique())]
weights /= weights.sum()
models = sorted(results["model"].unique())
tasks = list(weights.index)

# score matrix per task: rows = items, cols = models (paired design)
score_mats = {t: results[results.task == t].pivot(index="item_id", columns="model", values="score")[models].fillna(0.0)
              for t in tasks}


def composite(mats: dict[str, pd.DataFrame]) -> pd.Series:
    return sum(weights[t] * mats[t].mean() for t in tasks)


point = composite(score_mats)

rng = np.random.default_rng(SEED)
boot = np.empty((BOOTSTRAP_SAMPLES, len(models)))
arrays = {t: m.to_numpy() for t, m in score_mats.items()}
for b in range(BOOTSTRAP_SAMPLES):
    boot[b] = sum(weights[t] * a[rng.integers(0, len(a), len(a))].mean(axis=0) for t, a in arrays.items())
ci_lo, ci_hi = np.percentile(boot, 2.5, axis=0), np.percentile(boot, 97.5, axis=0)
boot_ranks = (-boot).argsort(axis=1).argsort(axis=1) + 1
p_first = (boot_ranks == 1).mean(axis=0)
median_rank = np.median(boot_ranks, axis=0)


def bradley_terry(mats: dict[str, pd.DataFrame], iters: int = 500) -> pd.Series:
    n = len(models)
    wins = np.zeros((n, n))
    for m in mats.values():
        a = m.to_numpy()
        for i in range(n):
            for j in range(i + 1, n):
                wins[i, j] += (a[:, i] > a[:, j]).sum() + 0.5 * (a[:, i] == a[:, j]).sum()
                wins[j, i] += (a[:, j] > a[:, i]).sum() + 0.5 * (a[:, i] == a[:, j]).sum()
    wins += 0.5 * (1 - np.eye(n))  # weak prior keeps undefeated / winless models finite
    games = wins + wins.T
    p = np.ones(n)
    for _ in range(iters):
        denom = (games / (p[:, None] + p[None, :])).sum(axis=1)
        p = wins.sum(axis=1) / denom
        p /= np.exp(np.log(p).mean())
    return pd.Series(1000 + 400 * np.log10(p), index=models)


bt = bradley_terry(score_mats)

agg = results.groupby("model").agg(
    items=("score", "size"), parse_rate=("parsed", "mean"), strict_json_rate=("strict_json", "mean"),
    error_rate=("error", lambda s: s.notna().mean()), latency_p50_s=("latency_s", "median"),
    cost_usd=("cost_usd", "sum"), output_tokens=("output_tokens", "mean"),
)
schema_rate = results[results.task == "log_extraction"].groupby("model")["schema_ok"].mean()

leaderboard = pd.DataFrame({
    "composite": point, "ci_low": ci_lo, "ci_high": ci_hi, "p_rank1": p_first, "median_rank": median_rank,
    "bt_rating": bt,
}, index=models).join(pd.DataFrame({t: score_mats[t].mean() for t in tasks})).join(agg)
leaderboard["schema_valid_rate"] = schema_rate
leaderboard["cost_per_100_items"] = leaderboard["cost_usd"] / leaderboard["items"] * 100
leaderboard = leaderboard.sort_values(["composite", "bt_rating"], ascending=False)
leaderboard.insert(0, "rank", range(1, len(leaderboard) + 1))

# Tiers: start a new tier when a model's CI upper bound falls below the tier leader's CI lower bound.
tier, tier_floor, tiers = 0, None, []
for _, row in leaderboard.iterrows():
    if tier_floor is None or row.ci_high < tier_floor:
        tier, tier_floor = tier + 1, row.ci_low
    tiers.append(tier)
leaderboard.insert(1, "tier", tiers)

show = ["rank", "tier", "composite", "ci_low", "ci_high", "p_rank1", "bt_rating", *tasks,
        "strict_json_rate", "schema_valid_rate", "error_rate", "latency_p50_s", "cost_per_100_items"]
leaderboard[show].round(3)

## Module 8 — Reporting

Three views of the same results: the ranked composite with its uncertainty, where each model wins or loses by task,
and what each point of quality costs.

In [ ]:
INK, MUTED, GRID, BAR = "#0b0b0b", "#898781", "#e1e0d9", "#2a78d6"
plt.rcParams.update({"font.size": 10, "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK, "xtick.color": MUTED,
                     "ytick.color": INK, "axes.spines.top": False, "axes.spines.right": False,
                     "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb"})

# 8.1 Leaderboard with bootstrap 95% CI
lb = leaderboard.iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 0.45 * len(lb) + 1.2))
ax.barh(lb.index, lb.composite, color=BAR, height=0.6)
ax.errorbar(lb.composite, lb.index, xerr=[lb.composite - lb.ci_low, lb.ci_high - lb.composite],
            fmt="none", ecolor=INK, elinewidth=1, capsize=3)
for y, (name, r) in enumerate(lb.iterrows()):
    ax.text(min(r.ci_high + 0.02, 1.02), y, f"{r.composite:.3f}  (tier {r.tier})", va="center", color=INK, fontsize=9)
ax.set_xlim(0, 1.2)
ax.set_xticks(np.linspace(0, 1, 6))
ax.set_xlabel("Composite score (weighted mean over tasks), 95% bootstrap CI")
ax.set_title("LLM ranking — structured data processing", loc="left", color=INK, fontsize=12)
ax.grid(axis="x", color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "leaderboard.png", dpi=150)
plt.show()

# 8.2 Model x task heatmap (single-hue sequential scale)
heat = leaderboard[tasks]
fig, ax = plt.subplots(figsize=(1.5 * len(tasks) + 3, 0.45 * len(heat) + 1.5))
im = ax.imshow(heat.to_numpy(), cmap="Blues", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(tasks)), [t.replace("_", "\n") for t in tasks])
ax.set_yticks(range(len(heat)), heat.index)
for (i, j), v in np.ndenumerate(heat.to_numpy()):
    ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=9, color="#ffffff" if v > 0.6 else INK)
ax.tick_params(length=0)
for s in ax.spines.values():
    s.set_visible(False)
fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label="Mean item score")
ax.set_title("Score by task", loc="left", color=INK, fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "task_heatmap.png", dpi=150)
plt.show()

# 8.3 Cost vs quality (only models with known, non-zero cost)
priced = leaderboard[leaderboard.cost_per_100_items.notna() & (leaderboard.cost_per_100_items > 0)]
if len(priced):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(priced.cost_per_100_items, priced.composite, s=60, color=BAR, edgecolor="#fcfcfb", linewidth=2, zorder=3)
    pareto, best = [], -1.0
    for name, r in priced.sort_values("cost_per_100_items").iterrows():
        if r.composite > best:
            pareto.append(name); best = r.composite
    for name, r in priced.iterrows():
        ax.annotate(name + (" ★" if name in pareto else ""), (r.cost_per_100_items, r.composite),
                    xytext=(6, 4), textcoords="offset points", fontsize=9, color=INK)
    ax.set_xscale("log")
    ax.set_xlabel("Cost per 100 items (USD, log scale)")
    ax.set_ylabel("Composite score")
    ax.set_title("Cost vs quality (★ = Pareto-optimal)", loc="left", color=INK, fontsize=12)
    ax.grid(color=GRID, linewidth=0.8)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "cost_vs_quality.png", dpi=150)
    plt.show()
else:
    print("No priced models in this run — cost/quality chart skipped.")

In [ ]:
# 8.4 Error analysis: lowest-scoring items per real model (baselines excluded)
real = results[results.provider != "baseline"]
if len(real):
    worst = real.sort_values("score").groupby("model").head(3)[["model", "task", "item_id", "score", "parsed", "error"]]
    display(worst)
else:
    print("Only calibration baselines ran. Set API keys (Module 4) to rank real models.")

# 8.5 Export
results.to_csv(RESULTS_DIR / "item_results.csv", index=False)
leaderboard.to_csv(RESULTS_DIR / "leaderboard.csv")
with open(RESULTS_DIR / "leaderboard.md", "w") as f:
    f.write("# LLM ranking — structured data processing\n\n")
    f.write(f"Data: {DATA_SOURCE}. Items: {len(ITEMS)} ({ITEMS_PER_TASK} per task). "
            f"Bootstrap samples: {BOOTSTRAP_SAMPLES}.\n\n")
    try:
        f.write(leaderboard[show].round(3).to_markdown())  # needs the optional `tabulate` package
    except ImportError:
        f.write("```\n" + leaderboard[show].round(3).to_string() + "\n```")
    f.write("\n")
print(f"Wrote results to {RESULTS_DIR.resolve()}")

## Interpreting and extending

* **Read tiers before ranks.** With 20 items per task, a few points of composite score are often within noise;
  raise `ITEMS_PER_TASK` until the CIs separate the models you care about.
* **Reliability matters as much as accuracy** for pipelines: check `strict_json_rate`, `schema_valid_rate` and
  `error_rate` alongside the composite.
* **Re-weighting** — set `TASK_WEIGHTS` to reflect your workload (e.g. weight `text_to_sql` higher for analytics use).
* **Adding a task** — write a generator returning `TaskItem`s and a scorer returning `(score, extras)`, then
  register both in `TASK_GENERATORS` and `SCORERS`. The ranking and reporting modules pick it up automatically.
* **Adding a provider** — any OpenAI-compatible endpoint can be added to `OPENAI_COMPATIBLE_PROVIDERS`; for other
  APIs, implement an adapter with a `complete(item) -> Completion` method and register it in `ADAPTERS`.
* **Per-model settings** — `ModelSpec.extra` is passed through to the API call, e.g.
  `{"output_config": {"effort": "low"}}` for a Claude model, to rank the same model at several effort levels.